# Clase 13 — MCP: el mismo agente, sin MCP y con MCP

**AI Agent Developer · ITBA · Laboratorio**

Partimos del agente de LangGraph de la Clase 12: recibe un email, `triage` lo clasifica, `knowledge` busca en la base de conocimiento, `writer` prepara un borrador y una persona lo revisa.

Hoy le cambiamos **una sola cosa**: cómo `knowledge` llega a la base de conocimiento. Primero lo corremos sin MCP. Después publicamos esa búsqueda en un server MCP y corremos el mismo agente otra vez.

> **Antes de empezar:** abrí esta notebook en la carpeta de trabajo de siempre, donde está `ai_agent_project`, y descomprimí ahí el ZIP de la clase, para que `agente_c13` quede al lado. Si te falta instalar algo, poné `RUN_INSTALLS=1` en el `.env` de esa carpeta y reiniciá el kernel. **No hace falta ninguna clave de API:** el LLM es opcional y aparece en la parte 7.

| Parte | Qué hacemos | En qué fijarse |
|---|---|---|
| 1 | Traer lo de clases anteriores | Que pase el control |
| 2 | Correr el agente **sin MCP** | Qué fuente encuentra `knowledge` |
| 3 | Crear el server MCP | Qué publica: Tool, Resource y Prompt |
| 4 | Correr el agente **con MCP** | Qué cambió y qué no |
| 5 | Hacer fallar la conexión | Que un error no es lo mismo que "no encontré" |
| 6 | Revisión humana | Quién decide, y que no se envía nada |
| 7 | LLM real (opcional) | Que el modelo no se entera de que hay MCP |
| 8 | Cierre | El JSON que se sube al campus |

Los fundamentos, los diagramas y las comparaciones están en el HTML de la clase.

## 1. Traer lo de clases anteriores

Estas tres celdas son las de siempre. Recuperan las rutas, comprueban que estén los archivos de las clases anteriores y dejan listo en `DEMO_TEXT` el email con el que trabajamos desde la Clase 7.

> El `runner` que imprime la primera celda es el de la Clase 12. El agente de hoy es el grafo que cargamos en la parte 2.

In [ ]:
# 0. Recuperar rutas y modos. No reescribimos archivos ni reportes de otras clases.
from pathlib import Path
import asyncio
import importlib
import importlib.util
import json
import os
try:
    from dotenv import load_dotenv
    load_dotenv(Path(".env"), override=False)
except ImportError:
    pass  # Variables definidas fuera del notebook siguen funcionando.
import socket
import subprocess
import sys
import time
from pprint import pprint

PROJECT_DIR = Path("ai_agent_project").resolve()
SRC_DIR = PROJECT_DIR / "src" / "ai_agent_course"
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MCP_DIR = PROJECT_DIR / "mcp"
TESTS_DIR = PROJECT_DIR / "tests"

for folder in (SRC_DIR, DATA_DIR, ARTIFACTS_DIR, MCP_DIR, TESTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").touch(exist_ok=True)
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

RUN_INSTALLS = os.getenv("RUN_INSTALLS", "0") == "1"
START_MCP_SERVER = os.getenv("START_MCP_SERVER", "0") == "1"
RUN_HTTP_CLIENT = os.getenv("RUN_HTTP_CLIENT", "0") == "1"
ENABLE_REAL_GMAIL = os.getenv("ENABLE_REAL_GMAIL", "0") == "1"
# Mismo nombre de flag y modelo que en C12: nunca convertir un fallo de API en un fake.
LIVE_MODE = os.getenv("LIVE_MODE", os.getenv("ENABLE_REAL_LLM", "0")) == "1"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
# Privacidad: el correo real de Gmail NO sale hacia un modelo cloud por accidente.
ALLOW_REAL_EMAIL_TO_LLM = os.getenv("ALLOW_REAL_EMAIL_TO_LLM", "0") == "1"
MCP_PORT = int(os.getenv("MCP_PORT", "8000"))
MCP_ENDPOINT = os.getenv("MCP_ENDPOINT", f"http://127.0.0.1:{MCP_PORT}/mcp")
MCP_INSTALLED = importlib.util.find_spec("mcp") is not None

print({"project": str(PROJECT_DIR), "mcp_sdk": MCP_INSTALLED,
       "runner": "class12_async_dag", "gmail_real_opt_in": ENABLE_REAL_GMAIL,
       "http_server_auto_start": START_MCP_SERVER,
       "llm_real": LIVE_MODE, "llm_model": GEMINI_MODEL if LIVE_MODE else "deshabilitado",
       "real_gmail_to_llm_allowed": ALLOW_REAL_EMAIL_TO_LLM})

In [ ]:
# Checkpoint fuerte: los archivos que vamos a REUTILIZAR existen y son legibles.
required = {
    "Caso de soporte C7": SRC_DIR / "support_case.py",
    "Dataset C7": DATA_DIR / "mock_gmail" / "threads.json",
    "KB C7": DATA_DIR / "mock_gmail" / "knowledge_base.md",
    "Human Gate C11": SRC_DIR / "hitl.py",
    "Orquestador C12": SRC_DIR / "multiagent.py",
    "Evidencia C12": ARTIFACTS_DIR / "class12_report.json",
}
missing = [name for name, path in required.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Falta el handoff: " + ", ".join(missing)
        + ". Ejecutá C7, C11 y C12 sobre ai_agent_project; C13 no fabrica datos previos."
    )

# Reutilizamos únicamente las interfaces realmente exportadas por la C12 del curso.
try:
    from ai_agent_course.support_case import load_support_case
    from ai_agent_course.multiagent import (
        WorkerTask, WorkerOutput, Evidence, run_supervisor_workers,
    )
    from ai_agent_course.hitl import (
        AgentRunState, prepare_action, apply_human_decision, JsonCheckpointStore,
    )
except ImportError as exc:
    raise RuntimeError(
        "Checkpoint incompatible: revisá la versión de multiagent.py / hitl.py. "
        "C13 necesita Evidence, WorkerTask, WorkerOutput, run_supervisor_workers y el Human Gate de C11. "
        "Reejecutá las versiones correspondientes de C11 y C12 en ai_agent_project."
    ) from exc

THREADS, KNOWLEDGE_BASE = load_support_case(PROJECT_DIR)
C12_REPORT = json.loads(required["Evidencia C12"].read_text(encoding="utf-8"))
assert isinstance(THREADS, list) and THREADS and isinstance(KNOWLEDGE_BASE, str)
assert C12_REPORT.get("class") == 12, "La evidencia no corresponde a C12"
print({"checkpoint": "OK", "emails_mock": len(THREADS),
       "orquestador": "ai_agent_course.multiagent", "etapa_anterior": C12_REPORT.get("architecture", {}).get("coordination")})

In [ ]:
# Inspeccionamos el mismo correo sintético de C7, sin publicar datos de Gmail real.
DEMO_THREAD = THREADS[0]
DEMO_TEXT = DEMO_THREAD["subject"] + ". " + " ".join(
    message["body"] for message in DEMO_THREAD["messages"]
)
print({"thread_id": DEMO_THREAD["thread_id"], "subject": DEMO_THREAD["subject"],
       "body_chars": len(DEMO_TEXT), "source": "class07_fixture"})

# El mismo contrato que aprendimos en C12 (nunca redefinimos los modelos).
example_task = WorkerTask(task_id="email-knowledge", role="knowledge",
    objective="Buscar evidencia de soporte con fuente verificable",
    success_criteria=["devolver evidencia o ausencia explícita"])
print(example_task.model_dump())

## 2. El agente sin MCP

El agente está en la carpeta `agente_c13`. Es el mismo que recibieron en la Clase 12, así que no lo reescribimos: lo importamos.

```text
                       ┌─ triage ─────┐
supervisor ─ dispatch ─┤              ├─ writer ─ review ─ finalize
                       └─ knowledge ──┘
```

| Nodo | Qué hace |
|---|---|
| `supervisor` | Decide qué especialistas trabajan |
| `dispatch` | Les reparte las tareas, en paralelo |
| `triage` | Clasifica el email y su prioridad |
| `knowledge` | Busca en la base de conocimiento con la tool `search_policy` |
| `writer` | Redacta el borrador con lo que encontraron los otros dos |
| `review` | Frena el grafo hasta que decida una persona |
| `finalize` | Cierra. No envía nada |

In [ ]:
# Cargamos el agente de la carpeta agente_c13. RUN_INSTALLS=1 autoriza la instalación.
AGENT_DIR = Path("agente_c13").resolve()
if not (AGENT_DIR / "app" / "graph.py").is_file():
    raise FileNotFoundError("Falta agente_c13: descomprimí el ZIP de la clase en esta carpeta.")
if RUN_INSTALLS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(AGENT_DIR / "requirements.txt")])
    importlib.invalidate_caches()
MCP_INSTALLED = importlib.util.find_spec("mcp") is not None
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

from app.contracts import SearchArgs # Pydantic model: define la `query` a buscar y la cantidad maxima (`limit`) de resultados
from app.graph import graph # el grafo de LangGraph
from app.tools import REGISTRY, search_policy # REGISTRY de las tools, y la tool `search_policy` especificamente

print("Nodos del grafo:", [nodo for nodo in graph.get_graph().nodes if not nodo.startswith("__")])
print("De dónde a dónde puede pasar:")
for arista in graph.get_graph().edges:
    print(f"  {arista.source} → {arista.target}")

Hoy, `knowledge` termina llamando a una función Python común: `search_policy`, que lee `agente_c13/data/knowledge_base.md`. Probémosla sola, con el email.

In [ ]:
# muestra la búsqueda como una función Python común, sin grafo, sin agente y sin MCP. Es la pieza que después vamos a mover al server
for encontrado in search_policy(SearchArgs(query=DEMO_TEXT, limit=2)):
    print(encontrado["source"], "| puntaje:", encontrado["score"])

### Aclaración

- No hay embeddings ni LLM: es una búsqueda por coincidencia de palabras, no entiende significado. Si el email dijera “comprobantes” en lugar de “facturas”, perdería puntaje. Para esta clase alcanza, porque el foco está en cómo se llega a la búsqueda.

Esta función corre el grafo e imprime lo que van haciendo los nodos. Corremos en modo `demo`: las decisiones son reglas, sin LLM, pero **el grafo y las tools son reales**.

El grafo siempre frena en `review`. Nadie aprueba nada hasta la parte 6.

Cada `▶` es la entrada a un nodo. Como `triage` y `knowledge` corren en paralelo, sus líneas se pueden intercalar.

In [ ]:
# Corre el grafo real e imprime por dónde pasa y qué hace cada nodo.
import uuid

EVENTOS = {"route_decided": "eligió a", "llm_called": "le preguntó a", "tool_called": "pidió la tool",
           "tool_returned": "recibió la respuesta de", "agent_completed": "terminó con",
           "agent_failed": "falló con"}

async def seguir_grafo(entrada, config):
    """Ejecuta el grafo e imprime un ▶ cada vez que entra a un nodo."""
    async for evento in graph.astream(entrada, config=config, stream_mode="custom"):
        actor, tipo, d = evento["actor"], evento["kind"], evento["details"]
        if tipo in ("node_started", "agent_started"):
            print("▶", actor)
        elif tipo in EVENTOS:
            detalle = d.get("tool") or d.get("selected_agents") or d.get("status") or d.get("error_type") or d.get("model")
            print(f"    {actor} {EVENTOS[tipo]} {detalle}")

async def correr_agente(email, modo="demo"):
    # config: seteamos un thread_id para poder retomarlo despues
    config = {"configurable": {"thread_id": uuid.uuid4().hex}}

    # init: armado del estado inicial del grafo
    inicial = {"request": email, 
               "mode": modo, # `demo` decide con reglas, mientras que `real` decide con LLM
               "routing_mode": "parallel", # `parallel` hace trabjar siempre a `triage` y `knowledge`
               "failure": "", # para simular una falla (vacío no simula)
               "requires_review": True, # en True hace el que grafo frene en `review`
               "worker_results": [], # lista vacia donde cada worker va a ir dejando sus resultados
               "trace": []} # lista vacía donde los nodos van anotando qué hicieron
    
    await seguir_grafo(inicial, config)
    estado = await graph.aget_state(config)
    print("■ El grafo quedó frenado en:", list(estado.next))
    return config, estado.values

def resumen(valores):
    """Qué hizo cada especialista y en qué estado quedó el borrador."""
    especialistas = {r["agent"]: {"status": r["status"], "error": r["error"],
                                  "fuentes": [e["source"] for e in r["evidence"]]}
                     for r in sorted(valores["worker_results"], key=lambda r: r["agent"])}
    return {**especialistas, "borrador": valores["draft"]["status"]}

In [ ]:
config_sin, sin_mcp = await correr_agente(DEMO_TEXT)
print('\n------------------------------------------------------------\n')
print(resumen(sin_mcp))
print('\n------------------------------------------------------------\n')
print(sin_mcp["draft"]["body"])
print('\n------------------------------------------------------------\n')
print(config_sin)

## 3. El server MCP

Un server MCP publica capacidades para que cualquier aplicación que hable el protocolo pueda usarlas. Este server **no reescribe la búsqueda**: llama a las mismas funciones de `app/tools.py`. Lo nuevo es el contrato que publica.

Publica las tres cosas que puede publicar un server. `knowledge` va a usar solo las Tools.

| Qué | Cuál | Para qué |
|---|---|---|
| Tool | `search_policy`, `read_section` | Una acción que se ejecuta |
| Resource | `support://kb` | Datos que se leen por su dirección |
| Prompt | `draft_support_reply` | Una plantilla de instrucciones |

In [ ]:
%%writefile agente_c13/kb_server.py
"""Server MCP de soporte: publica la base de conocimiento del agente. Solo lectura; no envía correos.

Desde la carpeta de la notebook:
  python agente_c13/kb_server.py
Atiende en http://127.0.0.1:8000/mcp por defecto.
"""
import os
from typing import Any

from mcp.server import MCPServer

from app import tools
from app.contracts import SearchArgs, SectionArgs

mcp = MCPServer("itba-support", instructions="Corpus de soporte sintético; solo lectura.")


@mcp.tool()
# @mcp.tool() significa “publicá esta función”. El SDK arma el contrato a partir de lo que escribimos
# por eso el nombre, los tipos y la descripción dejan de ser un simple `detalle`, porue es todo lo que ve quien se conecta
def search_policy(query: str, limit: int = 2) -> list[dict]:
    """Busca las secciones de la base de conocimiento que mejor responden a la consulta."""
    return tools.search_policy(SearchArgs(query=query, limit=limit))


@mcp.tool()
def read_section(heading: str) -> dict[str, Any]:
    """Devuelve una sección completa de la base de conocimiento a partir de su título."""
    return tools.read_section(SectionArgs(heading=heading))


@mcp.resource("support://kb")
def support_knowledge_base() -> str:
    """La base de conocimiento completa, para leerla sin hacer una búsqueda."""
    return (tools.DATA / "knowledge_base.md").read_text(encoding="utf-8")


@mcp.prompt()
def draft_support_reply(subject: str, citations: str) -> str:
    """Instrucciones que el host podría presentarle al modelo. No envía el email."""
    return (f"Prepará un borrador para el asunto {subject!r}. "
            f"Usá solamente la evidencia indicada: {citations}. "
            "Si falta evidencia, indicá la incertidumbre y pedí revisión humana. "
            "Nunca afirmes que el email fue enviado.")


if __name__ == "__main__":
    # El estado del agente queda en el host. Solo demo local: en Internet harían falta TLS y autenticación.
    try:
        mcp.run(transport="streamable-http", stateless_http=True, json_response=True,
                host="127.0.0.1", port=int(os.getenv("MCP_PORT", "8000")))
    except KeyboardInterrupt:
        print("Server apagado")

Primero lo probamos sin red: si al `Client` le pasamos el objeto del server, lo corre en este mismo proceso. Sirve para comprobar el contrato sin levantar nada y sin ningún LLM.

In [ ]:
# ¿Qué publica el server? Esto mismo es lo que descubre cualquier cliente.
from mcp import Client
import kb_server
importlib.reload(kb_server)  # Por si volviste a correr la celda anterior con cambios.

async with Client(kb_server.mcp) as client:
    tools_mcp = await client.list_tools()
    recursos = await client.list_resources()
    prompts = await client.list_prompts()

for tool in tools_mcp.tools:
    print(tool.name, "→", tool.description)
    print("   recibe:", list(tool.input_schema["properties"]))
print("Resources:", [str(r.uri) for r in recursos.resources])
print("Prompts:", [p.name for p in prompts.prompts])

In [ ]:
# Usamos cada una. La búsqueda tiene que devolver lo mismo que la función directa.
async with Client(kb_server.mcp) as client:
    busqueda = await client.call_tool("search_policy", {"query": DEMO_TEXT, "limit": 2})
    recurso = await client.read_resource("support://kb")
    prompt = await client.get_prompt("draft_support_reply",
                                     {"subject": DEMO_THREAD["subject"], "citations": "KB: Carga de facturas"})
    protocolo = str(client.protocol_version)

print("¿Hubo error?", busqueda.is_error)
print("¿Misma búsqueda?", busqueda.structured_content["result"] == search_policy(SearchArgs(query=DEMO_TEXT, limit=2)))
print("Resource:", recurso.contents[0].text[:40], "...")
print("Prompt:", prompt.messages[0].content.text)

MCP_INPROCESS = {"executed": True, "tools": [t.name for t in tools_mcp.tools],
                 "resources": [str(r.uri) for r in recursos.resources],
                 "prompts": [p.name for p in prompts.prompts], "protocol": protocolo}

Ahora lo levantamos **en otro proceso**, como si estuviera en otra máquina. Desde la notebook solo conocemos su URL.

Abrí una terminal en esta misma carpeta, con el mismo entorno de Python de la notebook, y corré:

```bash
python agente_c13/kb_server.py
```

Dejala abierta y a la vista: ahí vas a ver llegar cada pedido.

In [ ]:
# El mismo Client, ahora con una URL. El contrato es igual; cambia por dónde viaja.
SERVER_PROCESS, SERVER_STARTED_HERE = None, False  # Lo levantaste vos, desde la terminal.

def server_prendido():
    with socket.socket() as sock:
        return sock.connect_ex(("127.0.0.1", MCP_PORT)) == 0

if not server_prendido():
    raise RuntimeError(f"No hay ningún server en {MCP_ENDPOINT}. Levantalo en una terminal: python agente_c13/kb_server.py")

async with Client(MCP_ENDPOINT) as client:
    tools_http = await client.list_tools()

print("Tools por HTTP:", [t.name for t in tools_http.tools])
if "search_policy" not in [t.name for t in tools_http.tools]:
    raise RuntimeError(f"El server de {MCP_ENDPOINT} no es kb_server.py. Cerralo y levantá el de esta clase.")
HTTP_PROBE = {"executed": True, "transport": "streamable-http",
              "tools": [t.name for t in tools_http.tools]}

## 4. El agente con MCP

`knowledge` no llama a la búsqueda directamente. Le pide a un registro de tools que ejecute `search_policy`:

```text
Sin MCP   knowledge → REGISTRY → search_policy() → knowledge_base.md
Con MCP   knowledge → REGISTRY → Client → kb_server.py → search_policy() → knowledge_base.md
```

Si cambiamos qué hay detrás de ese nombre, ni `knowledge` ni el grafo se enteran. Por eso `app/graph.py` no cambia.

Para poder hacerlo, el agente de este ZIP trae dos agregados respecto del de la Clase 12:

```python
# app/agents.py: una llamada por MCP viaja por la red, y hay que esperarla
output=REGISTRY.execute(name,decision.tool_args,skill.allowed_tools)
if inspect.isawaitable(output):
    output=await output

# app/tools.py: cambiar quién ejecuta una tool, sin tocar su nombre, descripción ni schema
def replace_handler(self,name,handler):
    self._tools[name]=replace(self._tools[name],handler=handler)
```

`await` no hace que el server responda más rápido. Libera el programa mientras espera: mientras `knowledge` espera al server, `triage` sigue trabajando.

In [ ]:
# El lado cliente: misma entrada y misma salida que las tools locales.
MCP_TIMEOUT_S = 5.0

async def llamar_mcp(tool, args):
    """Llama a una tool del server. Un error nunca se convierte en "no encontré nada"."""
    async def llamar():
        async with Client(MCP_ENDPOINT) as client:
            return await client.call_tool(tool, args.model_dump())
    try:
        resultado = await asyncio.wait_for(llamar(), timeout=MCP_TIMEOUT_S)
    except asyncio.TimeoutError:
        raise
    except Exception as exc:
        raise ConnectionError(f"El server MCP no responde en {MCP_ENDPOINT}") from exc
    if resultado.is_error or resultado.structured_content is None:
        raise RuntimeError(f"La tool {tool} falló en el server")
    return resultado.structured_content

async def search_policy_por_mcp(args):
    return (await llamar_mcp("search_policy", args))["result"]

async def read_section_por_mcp(args):
    return await llamar_mcp("read_section", args)

Este es el cambio de toda la clase: dos líneas.

Vas a ver líneas `HTTP Request: POST ...`, a veces en rojo. No son errores: cada una es un mensaje que viajó al server. En la corrida sin MCP no aparecían. Mirá también la terminal: los mismos pedidos llegan al server.

In [ ]:
# Nombre, descripción y schema quedan iguales. Cambia quién ejecuta.
REGISTRY.replace_handler("search_policy", search_policy_por_mcp)
REGISTRY.replace_handler("read_section", read_section_por_mcp)

config_con, con_mcp = await correr_agente(DEMO_TEXT)
print()
print(resumen(con_mcp))

In [ ]:
# Mismo email, mismo grafo. ¿Cambió algo?
def nodos(valores):
    return sorted(paso["node"] for paso in valores["trace"])

def evidencia(valores):
    return next(r["evidence"] for r in valores["worker_results"] if r["agent"] == "knowledge")

COMPARACION = {"mismos_nodos": nodos(sin_mcp) == nodos(con_mcp),
               "misma_evidencia": evidencia(sin_mcp) == evidencia(con_mcp),
               "mismo_borrador": sin_mcp["draft"] == con_mcp["draft"]}
print(COMPARACION)

Si el resultado es el mismo, ¿qué ganamos? La búsqueda dejó de ser una función de este agente. Cualquier aplicación que hable MCP puede usarla sin copiar código, como este cliente, que no sabe nada de LangGraph:

In [ ]:
# Otro cliente, sin grafo ni agente, usa el mismo server.
async with Client(MCP_ENDPOINT) as otro:
    seccion = await otro.call_tool("read_section", {"heading": "Cambio de contraseña"})

print(seccion.structured_content)

## 5. Cuando algo falla

Con MCP aparece algo nuevo: la búsqueda ahora es una llamada que puede fallar. Comparemos tres casos. En los tres, `knowledge` termina sin evidencia, pero **no es lo mismo "no encontré" que "no pude buscar"**.

Antes de correr cada celda, predecí qué va a hacer `writer`.

In [ ]:
# Caso 1: el server responde, pero la base de conocimiento no tiene nada sobre esto.
_, sin_evidencia = await correr_agente("Tengo un reclamo y necesito conocer la política de devolución.")

In [ ]:
# Caso 2: el server está apagado. Antes de correr esta celda, apagalo con Ctrl+C en la terminal.
if server_prendido():
    raise RuntimeError("El server sigue prendido. Apagalo con Ctrl+C en la terminal y volvé a correr esta celda.")

_, server_caido = await correr_agente(DEMO_TEXT)

print()
print(server_caido["draft"]["body"])

In [ ]:
# Caso 3: el server tarda más de lo que estamos dispuestos a esperar. Antes, volvé a levantarlo en la terminal.
if not server_prendido():
    raise RuntimeError("El server está apagado. Levantalo en la terminal y volvé a correr esta celda.")

MCP_TIMEOUT_S = 0.001
_, con_timeout = await correr_agente(DEMO_TEXT)
MCP_TIMEOUT_S = 5.0

In [ ]:
# Los tres casos, lado a lado. En ninguno aparece una fuente inventada.
FALLAS = {"sin_evidencia": resumen(sin_evidencia), "server_caido": resumen(server_caido),
          "timeout": resumen(con_timeout)}

for caso, resultado in FALLAS.items():
    k = resultado["knowledge"]
    print(f'{caso:15}knowledge={k["status"]:9}error={str(k["error"]):17}fuentes={k["fuentes"]}  borrador={resultado["borrador"]}')

## 6. Revisión humana

La corrida de la parte 4 sigue frenada en `review`. MCP no cambia esto: **el server no aprueba nada**, y la decisión sigue en nuestra aplicación.

Probá también con `approve`. Para `edit` hace falta agregar `edited_text`. Si querés repetir la celda, antes volvé a correr el agente, porque esta corrida ya quedó cerrada.

In [ ]:
from langgraph.types import Command

pendiente = await graph.aget_state(config_con)
print("Próximo nodo:", pendiente.next)
print()
print(con_mcp["draft"]["body"])
print()

decision = {"decision": "reject", "reviewer": "laboratorio"}
await seguir_grafo(Command(resume=decision), config_con)
final = (await graph.aget_state(config_con)).values
print()
print(final["final"])

HITL_PROBE = {"paused_at": list(pendiente.next), "decision": decision["decision"],
              "status": final["final"]["status"], "sent": False}

## 7. LLM real (opcional)

Hasta acá no hubo ningún LLM. Con `LIVE_MODE=1` y una clave en el `.env`, `supervisor`, `triage`, `knowledge` y `writer` le preguntan al modelo. Consume tokens.

| Proveedor | En el `.env` |
|---|---|
| Gemini | `LLM_PROVIDER=gemini` y `GEMINI_API_KEY` |
| Claude | `LLM_PROVIDER=anthropic` y `ANTHROPIC_API_KEY` |
| DeepSeek | `LLM_PROVIDER=deepseek` y `DEEPSEEK_API_KEY` |

Esto es lo que el modelo ve de la tool. Es lo mismo con MCP y sin MCP, y por eso no se entera:

In [ ]:
print(json.dumps(REGISTRY.describe(["search_policy"])[0], ensure_ascii=False, indent=2))

In [ ]:
# Opcional: el modelo pide la tool, nuestro código la ejecuta y esa ejecución pasa por MCP.
LIVE_LLM_RESULT = {"executed": False, "reason": "LIVE_MODE=0"}
if LIVE_MODE:
    try:
        _, con_llm = await correr_agente(DEMO_TEXT, modo="real")
        print()
        print(con_llm["draft"]["body"])
        print()
        LIVE_LLM_RESULT = {"executed": True, **resumen(con_llm)}
    except Exception as exc:
        # Nunca convertimos un fallo de API en un resultado simulado.
        LIVE_LLM_RESULT = {"executed": False, "error": type(exc).__name__}
        print("El LLM no respondió:", exc)
print(LIVE_LLM_RESULT)

## 8. Cierre

**En parejas:** elegí una condición y cambiá **una sola cosa**.

- **A.** En `kb_server.py`, hacé que `search_policy` lance una excepción. Después volvé a correr esa celda y reiniciá el server en la terminal.
- **B.** En `kb_server.py`, agregá una demora de 2 segundos con `time.sleep` y bajá `MCP_TIMEOUT_S` a 1.
- **C.** Corré el agente con un email que incluya instrucciones maliciosas.

Antes de ejecutar, anotá tu predicción. Después contale al grupo:

1. qué pasó;
2. qué frontera falló: protocolo, contrato, política del host o conocimiento;
3. qué hicieron `writer` y la revisión humana.

**A2A** queda como ampliación conceptual en el HTML: esta notebook no construye ni consume un server A2A.

### Evidencia de cierre

Este es el JSON que se sube al campus. Registra qué recorridos se hicieron de verdad.

In [ ]:
# Reporte de recorridos realizados: no registra asuntos, cuerpos, IDs ni secretos.
report = {
    "class": 13,
    "stage": "mcp_interoperability",
    "project": "ai_agent_project",
    "source_checkpoint": {"class12_report_exists": True,
                          "c7_corpus_reused": True, "c12_contracts_imported": True},
    "mcp": {"inprocess": MCP_INPROCESS, "http": HTTP_PROBE, "sdk_installed": MCP_INSTALLED},
    "agent": {"engine": "langgraph", "sin_mcp": resumen(sin_mcp), "con_mcp": resumen(con_mcp),
              "comparacion": COMPARACION},
    "fail_closed": FALLAS,
    "hitl": HITL_PROBE,
    "llm_real": LIVE_LLM_RESULT,
    "a2a": "ampliacion_teorica_html",
}
REPORT_PATH = ARTIFACTS_DIR / "class13_report.json"
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
assert json.loads(REPORT_PATH.read_text(encoding="utf-8"))["class"] == 13
print("Reporte:", REPORT_PATH)
print({"mcp_inprocess": MCP_INPROCESS.get("executed"),
       "mcp_http": HTTP_PROBE.get("executed"),
       "llm_real": LIVE_LLM_RESULT.get("executed"),
       "hitl_rejected": HITL_PROBE["status"] == "cancelled"})

## Cierre TFI — Definition of Done verificable

- [ ] Recuperé `support_case.py`, el corpus de C7, `hitl.py` de C11 y `multiagent.py` de C12, **sin regenerarlos**.
- [ ] Distingo la función Python previa (`search_policy()`) del contrato publicado por MCP.
- [ ] Creé el Server con Tools, un Resource y un Prompt; expliqué quién los consume.
- [ ] Ejecuté `Client(kb_server.mcp)` con SDK real e inspeccioné los resultados.
- [ ] Levanté el Server en **otro proceso** y consumí su URL Streamable HTTP.
- [ ] `knowledge` obtuvo evidencia vía MCP, y el agente dio los mismos nodos, la misma evidencia y el mismo borrador que sin MCP.
- [ ] Pude provocar un server apagado y un timeout; el host no inventó evidencia.
- [ ] Distingo "no encontré" de "no pude buscar": sin evidencia, el borrador queda `partial` y no inventa referencias.
- [ ] La salida terminó en la revisión humana (`review`), sin enviar mensajes.
- [ ] **Opcional:** activé `LIVE_MODE=1` y probé un LLM real (no es requisito para las pruebas offline).
- [ ] `class13_report.json` registra qué recorridos se ejecutaron, y lo subí al campus.

**Desafío:** ¿en qué circunstancias **no** sumarías MCP y conservarías una Tool Python local? Justificá la decisión en términos de acoplamiento, operación y costo.

### Liberar el servidor local (si lo inició esta notebook)

Detener procesos evita conflictos de puertos si se vuelve a ejecutar. Si el servidor se inició en otra terminal, se detiene desde esa terminal.

In [ ]:
# Limpieza segura: sólo detenemos el proceso que esta notebook creó.
if SERVER_PROCESS is not None and SERVER_STARTED_HERE and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=3)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
    print("Servidor MCP docente detenido")
else:
    print("No se creó un servidor que haya que detener en este kernel")

## Continuidad del proyecto

Al finalizar C13 queda la integración MCP de la búsqueda de conocimiento: el server en `agente_c13/kb_server.py` y el reporte en `ai_agent_project/artifacts/class13_report.json`. En C14 ese sistema se expondrá por API y se desplegará; en C15 se medirán ejecuciones, errores, herramientas, costos y latencias.

**Qué tiene que quedar de C13:** un servidor MCP local con Tools, un Resource y un Prompt; un cliente en proceso y un cliente HTTP; un adaptador defensivo, `llamar_mcp`; el grafo de LangGraph sin cambios, y un borrador que termina en la revisión humana. El LLM real es un experimento optativo y no se declara ejecutado si no fue conectado.

> `llamar_mcp` y el cambio con `replace_handler` viven en esta notebook: no quedan guardados en un módulo. Para usarlos desde otra aplicación, hay que pasarlos a un archivo.